# Focus 
During extreme weather conditions, are different communal zip codes such as ones located in rural, suburban, or urban communities more accident prone 

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Load in the datatset 
df = pd.read_csv("../../US_Accidents_March23_sampled_500k.csv")

In [11]:
# Trim zipcodes 
df["Zipcode"] = df["Zipcode"].astype(str).str[:5]

# Check 
print(df["Zipcode"].str.len().value_counts()) 
print(df["Zipcode"].value_counts())

Zipcode
5.0    499884
Name: count, dtype: int64
Zipcode
27610    846
32819    788
33186    745
91761    739
70808    716
        ... 
31309      1
32346      1
88318      1
05743      1
84408      1
Name: count, Length: 17002, dtype: int64


##### Distinguishing commuities (what classifys a zipcode to be urban, suburban or rural)

In [12]:
# Road features 
road_features = ["Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal", "Turning_Loop"]

# Each feature in a zipcode 
zip_data = df.groupby("Zipcode")[road_features].mean()

# Nummber of accidents in a zipcode 
zip_data["accident_count"] = df.groupby("Zipcode").size()

# Removing zipcodes with less than 20 accidents 17002 --> 5333 zips 
zip_data = zip_data[zip_data["accident_count"] >= 20]

#Average of city-type features 
zip_data["urban_score"] = zip_data[["Amenity", "Crossing", "Station", "Stop", "Traffic_Signal"]].mean(axis=1)

print(zip_data["urban_score"].describe())

count    5333.000000
mean        0.061735
std         0.071064
min         0.000000
25%         0.011881
50%         0.038095
75%         0.085714
max         0.530000
Name: urban_score, dtype: float64


##### Grouping: Using 25% percentile as low cutoff and 75% as high cutoff 

In [13]:
# Grouping the different communities 
low_cutoff = 0.0119
high_cutoff = 0.0857

zip_data["Community"] = "Suburban" # middle ground 
zip_data.loc[zip_data["urban_score"] < low_cutoff, "Community"] = "Rural"
zip_data.loc[zip_data["urban_score"] >= high_cutoff, "Community"] = "Urban"

print(zip_data["Community"].value_counts())

Community
Suburban    2659
Urban       1340
Rural       1334
Name: count, dtype: int64


##### Grouping weather conditions to make it easier to define whats considered extreme weather 


In [14]:
# Weather conditions 
cond = pd.read_csv("../../hi.csv") # using Jasper's file
name = cond["Weather_Condition"].str.lower()

# Cut off for wind speed and visibility to clean impossible values 
df.loc[df["Wind_Speed(mph)"] > 100, "Wind_Speed(mph)"] = np.nan # 822 --> 83
df.loc[df["Visibility(mi)"] > 10, "Visibility(mi)"] = np.nan # 130 --> 10

# Intensity 
cond["is_light"] = name.str.contains("light|vicinity") # grouping variables without qualifiers 
cond["is_heavy"] = name.str.contains("heavy") | (
    name.str.contains("thunder|t-storm|tornado|funnel|squall|hail") # including those without qualifiers
    & ~name.str.contains("light|vicinity") # to avoid mislabeling rows (eg. Light Rain with Thunder) 
)

cond["is_moderate"] = (
    name.str.contains("rain|drizzle|shower|snow|sleet|wintry|ice pellets")
    & ~cond["is_light"] & ~cond["is_heavy"] # middle ground between light and heavy
)

# Rain / Snow 
cond["is_rain"] = name.str.contains("rain|showers|drizzle") # rain
cond["is_snow"] = name.str.contains("snow|wintry|sleet|ice pellets|hail|grains") # snow 
cond["is_freezing"] = name.str.contains("freezing|ice pellets|sleet|wintry") # cold 

# Storms 
cond["is_storm"]  = name.str.contains("thunder|t-storm|squall|tornado|funnel")

# Visibilty
cond["is_low_vis"] = name.str.contains("fog|haze|mist|smoke|dust|sand|volcanic|ash")

# Wind 
cond["is_windy"] = name.str.contains("windy|squall|tornado")

# Cloud Conditions 
cond["is_clouds"] = name.str.contains("fair|cloud|clear|overcast|scattered")

# Percepition (has NaN values)
cond["is_unknown"] = name.str.contains("n/a")

cond.sample(10)     

,Weather_Condition,count,is_light,is_heavy,is_moderate,is_rain,is_snow,is_freezing,is_storm,is_low_vis,is_windy,is_clouds,is_unknown
52,Blowing Snow,42,False,False,True,False,True,False,False,False,False,False,False
19,Snow,962,False,False,True,False,True,False,False,False,False,False,False
91,Light Blowing Snow,1,True,False,False,False,True,False,False,False,False,False,False
6,Light Rain,22532,True,False,False,True,False,False,False,False,False,False,False
27,Light Snow / Windy,401,True,False,False,False,True,False,False,False,True,False,False
51,Thunder / Windy,43,False,True,False,False,False,False,True,False,True,False,False
9,Fog,6396,False,False,False,False,False,False,False,True,False,False,False
8,Light Snow,8120,True,False,False,False,True,False,False,False,False,False,False
1,Mostly Cloudy,64877,False,False,False,False,False,False,False,False,False,True,False
89,Drifting Snow / Windy,1,False,False,True,False,True,False,False,False,True,False,False


In [ ]:
# Scoring intensity 1-4 to determine extreme weather 

cond["intensity"] = np.select(
    [cond["is_heavy"], cond["is_moderate"], cond["is_light"]],
    [3, 2, 1], 
    default=0 # covers non qualifiers (clear,cloudy,fair etc.) 
)

# Conditions that make driving more dangerous (eg. 0=none, 1=freezing, 2=storm,windy)
cond["add_hazards"] = (
    cond["is_freezing"].astype(int) + cond["is_storm"].astype(int)
    + cond["is_low_vis"].astype(int) + cond["is_windy"].astype(int)
)

# severity_score = raw total 
cond["severity_score"] = (cond["intensity"] + cond["add_hazards"]).astype(float)
cond.loc[cond["is_unknown"], "severity_score"] = np.nan    # unknown is not "mild"

s = cond["severity_score"]
cond["weather_level"] = np.select( #weather_level = totaled score
    [s >= 3, s == 2, s == 1, s == 0],
    [4, 3, 2, 1],         
    default=np.nan          
)

cond[["Weather_Condition", "intensity", "add_hazards", "severity_score", "weather_level"]].sample(20)

,Weather_Condition,intensity,add_hazards,severity_score,weather_level
23,Wintry Mix,2,1,3.0,4.0
59,Wintry Mix / Windy,2,2,4.0,4.0
39,Thunderstorms and Rain,3,1,4.0,4.0
79,Thunder / Wintry Mix,3,2,5.0,4.0
18,T-Storm,3,1,4.0,4.0
12,Fair / Windy,0,1,1.0,2.0
78,Light Snow with Thunder,1,1,2.0,3.0
103,Light Haze,1,1,2.0,3.0
46,T-Storm / Windy,3,2,5.0,4.0
51,Thunder / Windy,3,2,5.0,4.0
